In [ ]:
https://apis.data.go.kr/B551011/KorService2/detailIntro2?MobileOS=etc&MobileApp=Stay-Sync&contentId=contentId&contentTypeId=contentTypeId&serviceKey=SERVICE_KEY

In [1]:
import requests
import csv
import json
import pandas as pd
from IPython.display import display
import time

In [16]:
# SERVICE_KEY = '79eA4Jw2odSc97bzYRMnACC2TlJMPETbQmcHQaZaabL7xSMIXQzMcM8Q2xNuWGYbDuHWdL6/eTaSLsIWmgdr/A=='
# SERVICE_KEY = '2062d21d5b880264998e187ec135ed6d0e1a08ce3e90ca211fcde551104f4c72'
SERVICE_KEY = 'b4a80a272ee29e78a0af7af5b5e90e9366e3d5141979e3254d6be618599e3ad0'
# KBS_API_KEY=2062d21d5b880264998e187ec135ed6d0e1a08ce3e90ca211fcde551104f4c72
# KJY_API_KEY=79eA4Jw2odSc97bzYRMnACC2TlJMPETbQmcHQaZaabL7xSMIXQzMcM8Q2xNuWGYbDuHWdL6/eTaSLsIWmgdr/A==
# PJH_API_KEY=b4a80a272ee29e78a0af7af5b5e90e9366e3d5141979e3254d6be618599e3ad0

IOS = 'ios'
AND = 'android'
WEB = 'web'
ETC = 'etc'

MOBILE_OS = [IOS, AND, WEB, ETC]
INPUT_FILENAME = "../data/경기북부관광지.csv"
OUTPUT_FILENAME = "../detailIntro2.csv"

fieldnames = ['contentid','contenttypeid','heritage1','heritage2','heritage3','infocenter','opendate','restdate','expguide','expagerange','accomcount','useseason','usetime','parking','chkbabycarriage','chkpet','chkcreditcard']

In [3]:
def extract_items_from_api_response(data):
    """
    공공데이터 API 응답에서 item 목록을 안전하게 추출합니다.
    items가 dict, list, str, None 등 어떤 형태로 와도 오류 없이 처리합니다.
    """

    if not isinstance(data, dict):
        return []

    response = data.get("response", {})
    if not isinstance(response, dict):
        return []

    body = response.get("body", {})
    if not isinstance(body, dict):
        return []

    items_container = body.get("items", None)

    # 정상 케이스: "items": {"item": [...]}
    if isinstance(items_container, dict):
        items = items_container.get("item", [])

    # 혹시 list로 바로 오는 경우
    elif isinstance(items_container, list):
        items = items_container

    # "", None 등 데이터가 없는 경우
    else:
        items = []

    # item이 dict 하나로 오는 경우
    if isinstance(items, dict):
        return [items]

    # item이 list로 오는 경우
    if isinstance(items, list):
        return items

    return []

In [4]:
def normalize_id(value):
    """
    CSV에서 읽은 contentid, contenttypeid가 2846052.0처럼 float으로 들어오는 경우를 정리합니다.
    """

    if pd.isna(value):
        return None

    if isinstance(value, float):
        return str(int(value))

    return str(value).strip()

In [5]:
def fetch_detail_intro_to_df(
    content_pairs,
    save_each_step=True,
    save_final=False,
    checkpoint_filename="detailIntro2_temp_checkpoint.csv",
    sleep_seconds=0.1
):
    """
    contentId, contentTypeId 쌍을 사용하여 detailIntro2 API를 호출하고,
    중간에 오류가 발생하더라도 현재까지 수집된 데이터를 DataFrame으로 반환합니다.

    Parameters
    ----------
    content_pairs : list
        [(contentId, contentTypeId), ...] 형태
    save_each_step : bool
        True이면 호출 중간마다 checkpoint CSV 저장
    save_final : bool
        True이면 함수 내부에서 OUTPUT_FILENAME에 최종 저장
        기존 CSV에 이어붙일 경우 False 권장
    checkpoint_filename : str
        중간 저장 파일명
    sleep_seconds : float
        API 과호출 방지용 대기 시간

    Returns
    -------
    df : pandas.DataFrame
        현재까지 성공적으로 수집한 데이터
    failed_pairs : list
        요청 오류 또는 JSON 오류가 발생한 (contentId, contentTypeId) 목록
    empty_pairs : list
        API 응답은 정상이나 item 데이터가 없었던 목록
    """

    print(f"총 {len(content_pairs)}개의 contentId에 대해 API를 호출합니다.")

    all_items = []
    failed_pairs = []
    empty_pairs = []

    try:
        for idx, pair in enumerate(content_pairs, start=1):

            # pair 구조 방어 처리
            if not isinstance(pair, (list, tuple)) or len(pair) < 2:
                print(f"⚠️ 잘못된 pair 형식입니다: {pair}")
                failed_pairs.append(pair)
                continue

            contentId, contentTypeId = pair[0], pair[1]

            contentId = normalize_id(contentId)
            contentTypeId = normalize_id(contentTypeId)

            if not contentId or not contentTypeId:
                continue

            print(
                f"\n[{idx}/{len(content_pairs)}] "
                f"-> contentId: {contentId}, contentTypeId: {contentTypeId} 호출 중..."
            )

            API_URL = (
                f"https://apis.data.go.kr/B551011/KorService2/detailIntro2"
                f"?MobileOS={MOBILE_OS[3]}"
                f"&MobileApp=Stay-Sync"
                f"&_type=json"
                f"&contentId={contentId}"
                f"&contentTypeId={contentTypeId}"
                f"&serviceKey={SERVICE_KEY}"
            )

            try:
                response = requests.get(API_URL, timeout=10)
                response.raise_for_status()

                data = response.json()

                items = extract_items_from_api_response(data)

                if items:
                    all_items.extend(items)
                    print(f"   -> 성공적으로 {len(items)}개 항목을 가져왔습니다.")
                else:
                    print(f"   -> contentId {contentId}에 대한 항목이 없습니다.")
                    empty_pairs.append((contentId, contentTypeId))

                # 매 호출마다 임시 CSV 저장
                if save_each_step and all_items:
                    temp_df = pd.DataFrame(all_items)
                    temp_df = temp_df.reindex(columns=fieldnames)
                    temp_df.to_csv(
                        checkpoint_filename,
                        index=False,
                        encoding="utf-8-sig"
                    )

            except requests.exceptions.RequestException as e:
                print(
                    f"⚠️ 요청 오류 발생: "
                    f"contentId={contentId}, contentTypeId={contentTypeId}, error={e}"
                )
                failed_pairs.append((contentId, contentTypeId))
                continue

            except json.JSONDecodeError as e:
                print(
                    f"⚠️ JSON 변환 오류 발생: "
                    f"contentId={contentId}, contentTypeId={contentTypeId}, error={e}"
                )
                failed_pairs.append((contentId, contentTypeId))
                continue

            except Exception as e:
                print(
                    f"🚨 처리 중 오류 발생: "
                    f"contentId={contentId}, contentTypeId={contentTypeId}, error={e}"
                )
                failed_pairs.append((contentId, contentTypeId))
                continue

            # API 과호출 방지
            if sleep_seconds:
                time.sleep(sleep_seconds)

    except KeyboardInterrupt:
        print("\n⛔ 사용자가 실행을 중단했습니다.")
        print("현재까지 수집된 데이터로 DataFrame을 생성합니다.")

    finally:
        if not all_items:
            print("\n❌ 현재까지 수집된 유효 데이터가 없습니다.")
            return pd.DataFrame(columns=fieldnames), failed_pairs, empty_pairs

        df = pd.DataFrame(all_items)
        df = df.reindex(columns=fieldnames)

        if save_final:
            df.to_csv(OUTPUT_FILENAME, index=False, encoding="utf-8-sig")
            print(f"🎉 CSV 파일 '{OUTPUT_FILENAME}' 저장이 완료되었습니다.")

        print(f"\n✅ 현재까지 총 {len(df)}개의 데이터를 수집했습니다.")

        if failed_pairs:
            print(f"\n⚠️ 실패한 contentId 수: {len(failed_pairs)}개")
            print(failed_pairs[:20])

        if empty_pairs:
            print(f"\nℹ️ 항목이 없었던 contentId 수: {len(empty_pairs)}개")
            print(empty_pairs[:20])

        return df, failed_pairs, empty_pairs

In [16]:
df_input = pd.read_csv(INPUT_FILENAME)

required_cols = ["contentid", "contenttypeid"]

for col in required_cols:
    if col not in df_input.columns:
        raise ValueError(f"입력 파일에 '{col}' 컬럼이 없습니다.")

content_pairs = (
    df_input[["contentid", "contenttypeid"]]
    .dropna()
    .drop_duplicates()
    .values
    .tolist()
)

result_df, failed_pairs = fetch_detail_intro_to_df(
    content_pairs,
    save_each_step=True,
    save_final=False
)

combined_df = append_to_existing_csv(result_df)

display(combined_df)

총 3173개의 contentId에 대해 API를 호출합니다.

[1/3173] -> contentId: 3482667, contentTypeId: 12 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[2/3173] -> contentId: 129194, contentTypeId: 12 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[3/3173] -> contentId: 2894914, contentTypeId: 39 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[4/3173] -> contentId: 2831563, contentTypeId: 39 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[5/3173] -> contentId: 125462, contentTypeId: 12 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[6/3173] -> contentId: 2846052, contentTypeId: 39 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[7/3173] -> contentId: 2901496, contentTypeId: 39 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[8/3173] -> contentId: 798518, contentTypeId: 39 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[9/3173] -> contentId: 2627867, contentTypeId: 32 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[10/3173] -> contentId: 3045767, contentTypeId: 28 호출 중...
   -> 성공적으로 1개 항목을 가져왔습니다.

[11/3173] -> contentId: 2726655, contentTypeId: 28 호출 중...
   -> contentId 2726655에 대한 항목이 없습니다.

[12/3173

ValueError: too many values to unpack (expected 2)

In [6]:
def append_to_existing_csv(new_df, output_filename=OUTPUT_FILENAME):
    """
    새로 수집된 DataFrame을 기존 CSV에 이어붙이고,
    contentid 기준으로 중복 제거 후 저장합니다.
    """

    if new_df is None or new_df.empty:
        print("추가할 데이터가 없습니다.")

        try:
            existing_df = pd.read_csv(output_filename, dtype=str)
            return existing_df
        except FileNotFoundError:
            return pd.DataFrame(columns=fieldnames)

    try:
        existing_df = pd.read_csv(output_filename, dtype=str)
        print(f"기존 저장 데이터: {len(existing_df)}건")
    except FileNotFoundError:
        existing_df = pd.DataFrame(columns=fieldnames)
        print("기존 CSV 파일이 없어 새로 생성합니다.")

    new_df = new_df.astype(str)

    combined_df = pd.concat([existing_df, new_df], ignore_index=True)

    # contentid 기준 중복 제거
    if "contentid" in combined_df.columns:
        combined_df["contentid"] = combined_df["contentid"].astype(str)
        combined_df = combined_df.drop_duplicates(subset=["contentid"], keep="first")

    combined_df = combined_df.reindex(columns=fieldnames)

    combined_df.to_csv(output_filename, index=False, encoding="utf-8-sig")

    print(f"\n✅ 이어서 저장 완료: {output_filename}")
    print(f"최종 저장 데이터: {len(combined_df)}건")

    return combined_df

In [17]:
df_input = pd.read_csv(INPUT_FILENAME, dtype=str)

required_cols = ["contentid", "contenttypeid"]

for col in required_cols:
    if col not in df_input.columns:
        raise ValueError(f"입력 파일에 '{col}' 컬럼이 없습니다.")

content_pairs = (
    df_input[["contentid", "contenttypeid"]]
    .dropna()
    .drop_duplicates()
    .values
    .tolist()
)

print(f"호출 대상 contentId 수: {len(content_pairs)}개")

호출 대상 contentId 수: 3173개


In [7]:
result_df, failed_pairs, empty_pairs = fetch_detail_intro_to_df(
    content_pairs,
    save_each_step=True,
    save_final=False,
    checkpoint_filename="detailIntro2_temp_checkpoint.csv",
    sleep_seconds=0.1
)

combined_df = append_to_existing_csv(result_df)

display(combined_df)

NameError: name 'content_pairs' is not defined

In [8]:
retry_df, still_failed_pairs, retry_empty_pairs = fetch_detail_intro_to_df(
    failed_pairs,
    save_each_step=True,
    save_final=False,
    checkpoint_filename="detailIntro2_retry_checkpoint.csv",
    sleep_seconds=0.2
)

combined_df = append_to_existing_csv(retry_df)

display(combined_df)

NameError: name 'failed_pairs' is not defined

In [19]:
df_input = pd.read_csv(INPUT_FILENAME, dtype=str)

try:
    existing_df = pd.read_csv(OUTPUT_FILENAME, dtype=str)

    if "contentid" in existing_df.columns:
        saved_ids = set(existing_df["contentid"].dropna().astype(str))
    else:
        saved_ids = set()

except FileNotFoundError:
    saved_ids = set()

remaining_df = df_input[
    ~df_input["contentid"].astype(str).isin(saved_ids)
]

remaining_pairs = (
    remaining_df[["contentid", "contenttypeid"]]
    .dropna()
    .drop_duplicates()
    .values
    .tolist()
)

print(f"전체 입력 데이터: {len(df_input)}건")
print(f"이미 저장된 데이터: {len(saved_ids)}건")
print(f"남은 호출 대상: {len(remaining_pairs)}건")

전체 입력 데이터: 3173건
이미 저장된 데이터: 2218건
남은 호출 대상: 955건


In [20]:
remaining_result_df, failed_pairs, empty_pairs = fetch_detail_intro_to_df(
    remaining_pairs,
    save_each_step=True,
    save_final=False,
    checkpoint_filename="detailIntro2_remaining_checkpoint.csv",
    sleep_seconds=0.1
)

combined_df = append_to_existing_csv(remaining_result_df)

display(combined_df)

총 955개의 contentId에 대해 API를 호출합니다.

[1/955] -> contentId: 2726655, contentTypeId: 28 호출 중...
   -> contentId 2726655에 대한 항목이 없습니다.

[2/955] -> contentId: 2752670, contentTypeId: 28 호출 중...
   -> contentId 2752670에 대한 항목이 없습니다.

[3/955] -> contentId: 2738718, contentTypeId: 28 호출 중...
   -> contentId 2738718에 대한 항목이 없습니다.

[4/955] -> contentId: 2909629, contentTypeId: 28 호출 중...
   -> contentId 2909629에 대한 항목이 없습니다.

[5/955] -> contentId: 2361081, contentTypeId: 28 호출 중...
   -> contentId 2361081에 대한 항목이 없습니다.

[6/955] -> contentId: 2763412, contentTypeId: 28 호출 중...
   -> contentId 2763412에 대한 항목이 없습니다.

[7/955] -> contentId: 2752045, contentTypeId: 28 호출 중...
   -> contentId 2752045에 대한 항목이 없습니다.

[8/955] -> contentId: 2747505, contentTypeId: 28 호출 중...
   -> contentId 2747505에 대한 항목이 없습니다.

[9/955] -> contentId: 2909594, contentTypeId: 28 호출 중...
   -> contentId 2909594에 대한 항목이 없습니다.

[10/955] -> contentId: 2763770, contentTypeId: 28 호출 중...
   -> contentId 2763770에 대한 항목이 없습니다.

[11/

,contentid,contenttypeid,heritage1,heritage2,heritage3,infocenter,opendate,restdate,expguide,expagerange,accomcount,useseason,usetime,parking,chkbabycarriage,chkpet,chkcreditcard
0,2902578,39,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,2847366,39,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,2932046,14,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
3,2929100,38,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
4,2915110,38,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
528,1197233,28,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
529,1032754,28,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
530,1033306,28,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
531,1033333,28,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
